## Setup

In [2]:
import duckdb
from pathlib import Path

# Find this case's setup file from either the case folder or project root.
case_dir = Path.cwd()
if not (case_dir / "setup.sql").is_file():
    case_dir = case_dir / "Danny's Dinner"
setup_sql = (case_dir / "setup.sql").read_text(encoding="utf-8")

# Each notebook owns its database; rerunning this cell resets the sample data.
if "con" in globals():
    con.close()
con = duckdb.connect(":memory:")
con.execute(setup_sql)


In [3]:
# Query and display results.

print("dannys_diner.sales")
con.sql("""
    SELECT * FROM dannys_diner.sales
    limit 2;
""").show()

print("dannys_diner.menu")
con.sql("""
    SELECT * FROM dannys_diner.menu;
""").show()

print("dannys_diner.members")
con.sql("""
    SELECT * FROM dannys_diner.members;
""").show()

dannys_diner.sales
┌─────────────┬────────────┬────────────┐
│ customer_id │ order_date │ product_id │
│   varchar   │    date    │   int32    │
├─────────────┼────────────┼────────────┤
│ A           │ 2021-01-01 │          1 │
│ A           │ 2021-01-01 │          2 │
└─────────────┴────────────┴────────────┘

dannys_diner.menu
┌────────────┬──────────────┬───────┐
│ product_id │ product_name │ price │
│   int32    │   varchar    │ int32 │
├────────────┼──────────────┼───────┤
│          1 │ sushi        │    10 │
│          2 │ curry        │    15 │
│          3 │ ramen        │    12 │
└────────────┴──────────────┴───────┘

dannys_diner.members
┌─────────────┬────────────┐
│ customer_id │ join_date  │
│   varchar   │    date    │
├─────────────┼────────────┤
│ A           │ 2021-01-07 │
│ B           │ 2021-01-09 │
└─────────────┴────────────┘



## 1. What is the total amount each customer spent at the restaurant?

In [4]:
con.sql("""
    SELECT s.customer_id, 
     sum(price) as total_spent
       FROM 
    dannys_diner.sales  as s left join dannys_diner.menu as m 
    on s.product_id = m.product_id

    group by s.customer_id 
    order by 1 asc 

""").show()

┌─────────────┬─────────────┐
│ customer_id │ total_spent │
│   varchar   │   int128    │
├─────────────┼─────────────┤
│ A           │          76 │
│ B           │          74 │
│ C           │          36 │
└─────────────┴─────────────┘



## 2. How many days has each customer visited the restaurant?

In [5]:
con.sql("""
    SELECT
      customer_id, 
     count(distinct order_date) as days
       FROM 
    dannys_diner.sales 

    group by 1 
    order by 1 asc 

""").show()

┌─────────────┬───────┐
│ customer_id │ days  │
│   varchar   │ int64 │
├─────────────┼───────┤
│ A           │     4 │
│ B           │     6 │
│ C           │     2 │
└─────────────┴───────┘



## 3. What was the first item from the menu purchased by each customer?

In [6]:
con.sql("""
    
    select  customer_id, product_name as first_product_name from
    
        (SELECT s.customer_id, 
        rank() over (partition by s.customer_id order by order_date asc ) as rank, 
        product_name 
        FROM 
        dannys_diner.sales  as s left join dannys_diner.menu as m 
        on s.product_id = m.product_id)

    where rank = 1
    order by 1 


""").show()

┌─────────────┬────────────────────┐
│ customer_id │ first_product_name │
│   varchar   │      varchar       │
├─────────────┼────────────────────┤
│ A           │ sushi              │
│ A           │ curry              │
│ B           │ curry              │
│ C           │ ramen              │
│ C           │ ramen              │
└─────────────┴────────────────────┘



## 4. What is the most purchased item on the menu and how many times was it purchased by all customers?

In [7]:
con.sql("""
    SELECT 
    m.product_name,
     count(*) as total_sales,
       FROM 
    dannys_diner.sales  as s left join dannys_diner.menu as m 
    on s.product_id = m.product_id

    group by 1
    order by total_sales desc 
    limit 1 

""").show()

┌──────────────┬─────────────┐
│ product_name │ total_sales │
│   varchar    │    int64    │
├──────────────┼─────────────┤
│ ramen        │           8 │
└──────────────┴─────────────┘



In [8]:
con.sql("""
    SELECT 
    s.customer_id,
    m.product_name,
    
     count(*) as total_sales,
       FROM 
    dannys_diner.sales  as s left join dannys_diner.menu as m 
    on s.product_id = m.product_id

    where product_name in ('ramen')
    group by 1,2
    order by 1 asc 

""").show()

┌─────────────┬──────────────┬─────────────┐
│ customer_id │ product_name │ total_sales │
│   varchar   │   varchar    │    int64    │
├─────────────┼──────────────┼─────────────┤
│ A           │ ramen        │           3 │
│ B           │ ramen        │           2 │
│ C           │ ramen        │           3 │
└─────────────┴──────────────┴─────────────┘



## 5. Which item was the most popular for each customer?

In [9]:
con.sql("""

select * from (
select *, rank() over (partition by customer_id order by total_sales desc) as rank     

from 
    (
    SELECT 
    s.customer_id,
    m.product_name,
    
     count(*) as total_sales,
       FROM 
    dannys_diner.sales  as s left join dannys_diner.menu as m 
    on s.product_id = m.product_id

    group by 1,2 )) 
    where rank = 1
    order by 1 asc

""").show()

┌─────────────┬──────────────┬─────────────┬───────┐
│ customer_id │ product_name │ total_sales │ rank  │
│   varchar   │   varchar    │    int64    │ int64 │
├─────────────┼──────────────┼─────────────┼───────┤
│ A           │ ramen        │           3 │     1 │
│ B           │ sushi        │           2 │     1 │
│ B           │ ramen        │           2 │     1 │
│ B           │ curry        │           2 │     1 │
│ C           │ ramen        │           3 │     1 │
└─────────────┴──────────────┴─────────────┴───────┘



## 6. Which item was purchased first by the customer after they became a member?

In [10]:
con.sql("""

select * from (
    SELECT 
    s.customer_id,
    order_date, 
    join_date,
    m.product_name,
    rank() over (partition by s.customer_id order by order_date asc ) as rank
    FROM 
    dannys_diner.sales  as s left join dannys_diner.menu as m 
    on s.product_id = m.product_id

    left join dannys_diner.members as mem
    on s.customer_id = mem.customer_id

    where  s.order_date >= mem.join_date
    order by 1 asc)

    where rank = 1 

""").show()

┌─────────────┬────────────┬────────────┬──────────────┬───────┐
│ customer_id │ order_date │ join_date  │ product_name │ rank  │
│   varchar   │    date    │    date    │   varchar    │ int64 │
├─────────────┼────────────┼────────────┼──────────────┼───────┤
│ A           │ 2021-01-07 │ 2021-01-07 │ curry        │     1 │
│ B           │ 2021-01-11 │ 2021-01-09 │ sushi        │     1 │
└─────────────┴────────────┴────────────┴──────────────┴───────┘



## 7. Which item was purchased just before the customer became a member?

In [11]:
con.sql("""

select * from (
    SELECT 
    s.customer_id,
    order_date, 
    join_date,
    m.product_name,
    rank() over (partition by s.customer_id order by order_date desc ) as rank
    FROM 
    dannys_diner.sales  as s left join dannys_diner.menu as m 
    on s.product_id = m.product_id

    left join dannys_diner.members as mem
    on s.customer_id = mem.customer_id

    where  s.order_date < mem.join_date
    order by 1 asc)

   where rank = 1 

""").show()

┌─────────────┬────────────┬────────────┬──────────────┬───────┐
│ customer_id │ order_date │ join_date  │ product_name │ rank  │
│   varchar   │    date    │    date    │   varchar    │ int64 │
├─────────────┼────────────┼────────────┼──────────────┼───────┤
│ A           │ 2021-01-01 │ 2021-01-07 │ sushi        │     1 │
│ A           │ 2021-01-01 │ 2021-01-07 │ curry        │     1 │
│ B           │ 2021-01-04 │ 2021-01-09 │ sushi        │     1 │
└─────────────┴────────────┴────────────┴──────────────┴───────┘



## 8. What is the total items and amount spent for each member before they became a member?

In [12]:
con.sql("""


    SELECT 
    s.customer_id,
    count(s.product_id) as total_products,
    sum(price) as total_spent
    FROM 
    dannys_diner.sales  as s 
    left join dannys_diner.menu as m 
    on s.product_id = m.product_id

    left join dannys_diner.members as mem
    on s.customer_id = mem.customer_id

    where  s.order_date < mem.join_date
    group by 1 
    order by 1 asc

""").show()

┌─────────────┬────────────────┬─────────────┐
│ customer_id │ total_products │ total_spent │
│   varchar   │     int64      │   int128    │
├─────────────┼────────────────┼─────────────┤
│ A           │              2 │          25 │
│ B           │              3 │          40 │
└─────────────┴────────────────┴─────────────┘



## 9.  If each $1 spent equates to 10 points and sushi has a 2x points multiplier - how many points would each customer have?

In [13]:
con.sql("""


    SELECT 
    s.customer_id,
    sum(points) as total_points
    FROM 
    dannys_diner.sales  as s 
    left join 
    
    (select *, case when product_name = 'sushi' then 2*10*price else 10*price end as points
      
        from
    dannys_diner.menu) as m 
    on s.product_id = m.product_id

    group by 1 
    order by 1 asc

""").show()

┌─────────────┬──────────────┐
│ customer_id │ total_points │
│   varchar   │    int128    │
├─────────────┼──────────────┤
│ A           │          860 │
│ B           │          940 │
│ C           │          360 │
└─────────────┴──────────────┘



## 10. In the first week after a customer joins the program (including their join date) they earn 2x points on all items, not just sushi - how many points do customer A and B have at the end of January?

In [14]:
con.sql("""


SELECT 
    s.customer_id,
    sum(Case when s.order_date >= mem.join_date and s.order_date < mem.join_date + interval '7 days' then 2*10*price 
         when m.product_name = 'sushi' then 2*10*price
         else 10*price end) as points

    FROM 
    dannys_diner.sales  as s left join dannys_diner.menu as m 
    on s.product_id = m.product_id

    join dannys_diner.members as mem
    on s.customer_id = mem.customer_id

    where  s.order_date >= DATE '2021-01-01'
    AND s.order_date < DATE '2021-02-01'

    group by 1 
    order by 1 asc 

""").show()

┌─────────────┬────────┐
│ customer_id │ points │
│   varchar   │ int128 │
├─────────────┼────────┤
│ A           │   1370 │
│ B           │    820 │
└─────────────┴────────┘

